# Affinage — apprendre à écrire l'hésitation

Le plancher mesuré : `wav2vec2-large-960h-lv60-self` rend **0 remplissage sur 7**.
Il n'écrit pas un mot faux, il n'écrit rien — il n'a jamais vu de transcription qui
en contienne une. Son vocabulaire est des lettres, donc `UH` est déjà écrivable :
il n'y a pas de vocabulaire à changer, seulement une habitude à prendre.

AMI apporte ça : 100 h de réunions en CC-BY-4.0, locuteurs **majoritairement non
natifs**, transcrites en majuscules sans ponctuation avec les hésitations écrites
— exactement le format de sortie du modèle.

`FULL = False` n'entraîne que la tête, ce qui tient dans une session courte.
`FULL = True` entraîne tout le réseau, extracteur convolutionnel excepté.

Réglages Kaggle : GPU **T4**, Internet **on**, persistance **Files only** (le
modèle affiné se garde dans `/kaggle/working`).

Ce qui rentre à la maison : `finetuned.json`, quelques kilo-octets. Les poids
restent ici tant qu'aucun run n'a gagné.

In [ ]:
MODEL = "facebook/wav2vec2-large-960h-lv60-self"

FULL = False          # True: tout le réseau, extracteur convolutionnel excepté
UTTERANCES = 20000    # tirées d'AMI en flux, soit une quinzaine d'heures
EPOCHS = 2
LR = 1e-4 if FULL else 3e-4
BATCH = 4 if FULL else 8
AMP = False           # demi-précision : wav2vec2-large y est réputé instable
MAX_SECONDS = 15      # au-delà, une prise ne tient pas en mémoire avec ce modèle
RUN = "full" if FULL else "head"

In [ ]:
import json, math, pathlib, random

import soundfile
import torch
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(MODEL)
model = Wav2Vec2ForCTC.from_pretrained(MODEL, ctc_loss_reduction="mean")
# `masked_spec_embed` is absent from some released checkpoints, and transformers
# fills a missing parameter with NaN; SpecAugment then poisons every train-mode
# forward, including one whose encoder is frozen. Give it its intended init.
with torch.no_grad():
    if model.wav2vec2.masked_spec_embed.isnan().any():
        model.wav2vec2.masked_spec_embed.uniform_()
        print("masked_spec_embed was NaN on load -- initialised")
poisoned = [name for name, p in model.named_parameters() if p.isnan().any()]
if poisoned:
    raise SystemExit(f"NaN parameters after loading: {poisoned}")
# The convolutional feature extractor stays frozen in every regime -- the recipe
# wav2vec2 was released with, and what keeps the signal contract of the export.
model.freeze_feature_encoder()
if not FULL:
    for parameter in model.wav2vec2.parameters():
        parameter.requires_grad = False
else:
    model.gradient_checkpointing_enable()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{RUN}: {trainable:,} trainable of {sum(p.numel() for p in model.parameters()):,}"
      f" on {device}")

In [ ]:
# Streamed rather than downloaded: the config is 29 GB and we want a slice of it.
stream = load_dataset("edinburghcstr/ami", "ihm", split="train", streaming=True)

# AMI spells letters out with stops -- "S. S. H." -- and the vocabulary is
# letters and an apostrophe, nothing else. Left alone, every stop becomes an
# `<unk>` the network is taught to emit. Anything outside the vocabulary becomes
# a space instead, which is what the mouth said anyway.
KEPT = {token for token in processor.tokenizer.get_vocab() if len(token) == 1} - {"|"}


def clean(text):
    return " ".join("".join(c if c in KEPT else " " for c in text).split())


corpus, hesitant, mended = [], 0, 0
for row in stream:
    if len(corpus) >= UTTERANCES:
        break
    text = clean(row["text"].strip())
    mended += text != row["text"].strip()
    audio = row["audio"]["array"]
    seconds = len(audio) / row["audio"]["sampling_rate"]
    if not text or seconds < 0.4 or seconds > MAX_SECONDS:
        continue
    corpus.append({"audio": audio.astype("float32"), "text": text, "seconds": seconds})
    hesitant += any(word in ("UH", "UM") for word in text.split())

hours = sum(e["seconds"] for e in corpus) / 3600
print(f"{len(corpus)} utterances, {hours:.1f} h, {hesitant} carrying a filled pause"
      f" ({100 * hesitant / len(corpus):.0f} %), {mended} with a symbol outside the vocabulary")
print("sample:", corpus[0]["text"])

In [ ]:
def lots(entries, size):
    """Group by length: padding a batch to its longest member is cheapest when
    the members are of a kind. The groups are shuffled per epoch, so the saving
    stays and the fixed short-to-long curriculum does not."""
    ordered = sorted(entries, key=lambda e: e["seconds"])
    return [ordered[i:i + size] for i in range(0, len(ordered), size)]


def batched(lot):
    heard = processor([e["audio"] for e in lot], sampling_rate=16000,
                      padding=True, return_tensors="pt")
    written = processor(text=[e["text"] for e in lot], padding=True,
                        return_tensors="pt")
    lengths = written.attention_mask.sum(dim=-1)
    return heard, written.input_ids, lengths

In [ ]:
# One batch, taken apart. A NaN has three places to be born -- the signal, the
# forward, the loss -- and reading which one costs a second here instead of an
# afternoon of guessing.
heard, targets, target_lengths = batched(lots(corpus, BATCH)[len(corpus) // BATCH // 2])
values = heard.input_values.to(device)
mask = heard.attention_mask.to(device)
print("signal      NaN:", values.isnan().any().item(),
      " range:", f"{values.min():.2f}..{values.max():.2f}")

model.eval()
for half in (False, True):
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16,
                                         enabled=half and device == "cuda"):
        logits = model(values, attention_mask=mask).logits
    print(f"logits fp{'16' if half else '32'} NaN:", logits.isnan().any().item(),
          " range:", f"{logits.float().min():.1f}..{logits.float().max():.1f}")

model.train()
with torch.no_grad():
    logits = model(values, attention_mask=mask).logits
print("logits train-mode NaN:", logits.isnan().any().item())

lengths = model._get_feat_extract_output_lengths(mask.sum(dim=-1)).to(torch.long)
print("frames:", lengths.tolist())
print("targets:", target_lengths.tolist(), " max id:", targets.max().item(),
      "of", model.config.vocab_size)
print("targets shorter than frames:", bool((target_lengths.to(device) <= lengths).all()))

# The forward is clean, so the loss is where it is born. Two suspects: lengths
# handed to the loss on two different devices, and a batch of the shortest
# utterances, which is what the sorted groups put first and what the training
# loop actually met at step zero.
def losses(lot, half):
    """Eval mode, always: SpecAugment masks frames at random, so a probe that
    left it on would read a different draw at every call and report the spread
    of the augmentation as though it were the spread of the thing measured."""
    model.eval()
    heard, targets, target_lengths = batched(lot)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16,
                                         enabled=half and device == "cuda"):
        logits = model(heard.input_values.to(device),
                       attention_mask=heard.attention_mask.to(device)).logits
    frames = torch.nn.functional.log_softmax(logits.float(), dim=-1)
    lengths = model._get_feat_extract_output_lengths(
        heard.attention_mask.sum(dim=-1)).to(torch.long)
    return torch.nn.functional.ctc_loss(
        frames.transpose(0, 1),
        targets.to(device), input_lengths=lengths.cpu(),
        target_lengths=target_lengths.cpu(),
        blank=model.config.pad_token_id, reduction="none", zero_infinity=True)


groups = lots(corpus, BATCH)
for name, lot in (("shortest", groups[0]), ("middle", groups[len(groups) // 2])):
    for half in (False, True):
        each = losses(lot, half)
        print(f"{name:<9} forward fp{'16' if half else '32'}:",
              [round(v, 2) for v in each.tolist()])
print("blank id:", model.config.pad_token_id, " shortest:",
      f"{groups[0][0]['seconds']:.2f}s", repr(groups[0][0]["text"][:60]))

In [ ]:
groups = lots(corpus, BATCH)
updates = len(groups) * EPOCHS
ramp = max(1, round(0.1 * updates))
optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
schedule = torch.optim.lr_scheduler.LambdaLR(
    optimiser,
    lambda u: (u + 1) / ramp if u < ramp else max(0.0, (updates - u) / max(1, updates - ramp)))
half = AMP and device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=half)
blank = model.config.pad_token_id
print(f"{len(groups)} batches per epoch, {updates} steps, {ramp} warming up")

model.train()
step = 0
for epoch in range(EPOCHS):
    random.Random(epoch).shuffle(groups)
    for lot in groups:
        heard, targets, target_lengths = batched(lot)
        with torch.autocast("cuda", dtype=torch.float16, enabled=half):
            logits = model(heard.input_values.to(device),
                           attention_mask=heard.attention_mask.to(device)).logits
        # Everything from here down stays in fp32, autocast or not: a CTC loss
        # over log-probabilities is where reduced precision turns into a silent
        # NaN rather than an error. Learnt twice now, in this project.
        frames = torch.nn.functional.log_softmax(logits.float(), dim=-1)
        input_lengths = model._get_feat_extract_output_lengths(
            heard.attention_mask.sum(dim=-1)).to(torch.long)
        loss = torch.nn.functional.ctc_loss(
            frames.transpose(0, 1), targets.to(device),
            input_lengths=input_lengths, target_lengths=target_lengths,
            blank=blank, reduction="mean", zero_infinity=True)
        if not torch.isfinite(loss):
            # The batch of a failing step is what the probes could not reach:
            # the groups are shuffled, so step zero is not the first group. Say
            # what it held rather than only that it failed.
            print(f"loss is {loss.item()} at step {step}")
            print("  frames :", input_lengths.tolist())
            print("  targets:", target_lengths.tolist())
            print("  logits NaN:", logits.isnan().any().item(),
                  " range:", f"{logits.float().min():.1f}..{logits.float().max():.1f}")
            for entry in lot:
                print(f"  {entry['seconds']:.2f}s {entry['text'][:70]!r}")
            raise SystemExit("stopping rather than training on nothing")
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimiser)
        scaler.update()
        schedule.step()
        optimiser.zero_grad(set_to_none=True)
        step += 1
        if step % 50 == 0 or step == 1:
            print(f"epoch {epoch} step {step}/{updates} loss {loss.item():.3f}"
                  f" lr {schedule.get_last_lr()[0]:.2e}")

In [ ]:
# Found rather than guessed: Kaggle nests a mounted dataset at a depth
# that is not the same from one attachment to the next.
root = next(pathlib.Path("/kaggle/input").rglob("references.json")).parent
references = json.loads((root / "references.json").read_text(encoding="utf-8"))
asked = json.loads((root / "answers" / "asked.json").read_text(encoding="utf-8"))

model.eval()
rows = []
for wav in sorted(root.rglob("*.wav")):
    which, slug = wav.parent.name, wav.stem
    audio, rate = soundfile.read(wav)
    heard = processor(audio, sampling_rate=rate, return_tensors="pt")
    with torch.no_grad():
        logits = model(heard.input_values.to(device),
                       attention_mask=heard.attention_mask.to(device)).logits
    text = processor.batch_decode(logits.argmax(dim=-1))[0]
    row = {"set": which, "slug": slug, "verbatim": text}
    if which == "stumbles":
        row["reference"] = references[slug]["text"]
        row["kind"] = references[slug]["kind"]
    else:
        row["asked"] = asked.get(slug)
    rows.append(row)
    print(f"\n[{which}/{slug}]")
    if "reference" in row:
        print(f"  attendu : {row['reference']}")
    print(f"  rendu   : {text}")

In [ ]:
out = pathlib.Path(f"/kaggle/working/finetuned-{RUN}.json")
out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(out, out.stat().st_size, "bytes")

# The weights stay here until a run has won something; only the reading travels.
model.save_pretrained(f"/kaggle/working/{RUN}")
processor.save_pretrained(f"/kaggle/working/{RUN}")
print("weights kept in", f"/kaggle/working/{RUN}")